# Model author: declare and inspect one simple resonator

This Notebook is for the person who owns the reusable circuit model. It starts at `CircuitPlan`, names every electrical boundary once, then declares Direct solve, physical-quantity, and report requests. At the current `CONVERGING` checkpoint the package is an API-only scaffold, so executing a construction cell intentionally raises `ScaffoldUnavailableError`.

In [ ]:
from scnsim import (
    CircuitPlan, CircuitRun, DiagonalRootSpec, DirectSolveSpec,
    ReportSpec, SParameterTrace, library as sc, units as u,
)

## 1. The Plan is the physical authority

Components come from an exact Library. Pins are assigned once to named nets or the Plan's one reference. The external port binds a net, not a component pin.

In [ ]:
plan = CircuitPlan(id="simple_readout")

coupling_cap = plan.add(
    sc.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
readout = plan.add(
    sc.grounded_parallel_linear_lc_resonator(
        id="readout",
        subsystem_capacitance=110.0 * u.fF,
        inductance=5.8 * u.nH,
    )
)

plan.reference("ground")
plan.net("signal_in", coupling_cap.pin("a"))
plan.net(
    "readout_node",
    coupling_cap.pin("b"),
    readout.pin("signal"),
)
plan.add_port(
    id="signal_in",
    at="signal_in",
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

## 2. The Run owns execution; the Spec explains the requested operation

`DirectSolveSpec` asks for the S/Y/Z response over a grid. `SParameterTrace` only names a projection of that complete matrix; it is not another solve. `run.explain(...)` is the preflight inspection point.

In [ ]:
run = CircuitRun(plan=plan, workspace="results/simple_readout")
view = run.original
frequency_grid = [5.5, 5.6, 5.7, 5.8, 5.9, 6.0, 6.1, 6.2, 6.3] * u.GHz

direct_spec = DirectSolveSpec(
    frequencies=frequency_grid,
    traces=(
        SParameterTrace(
            id="reflection",
            input_port="signal_in", input_mode=(),
            output_port="signal_in", output_mode=(),
        ),
    ),
)
run.explain(view, direct_spec).show()
direct = run.solve(view, direct_spec)
direct.s.show(magnitude="db")

## 3. Evaluate one physical quantity without inventing a sweep heuristic

`DiagonalRootSpec` asks for the anchored complex root of one named dynamic-operator diagonal. It is appropriate for a local/bare coordinate. A coupled retained block would use `HybridizedPoleSpec`; the full labeled matrix would use `OperatorSpec`.

In [ ]:
readout_root = DiagonalRootSpec(
    coordinate="readout_node",
    anchor=6.2 * u.GHz,
)
evaluated = run.evaluate(view, readout_root)
evaluated.show()
evaluated.frequency.to(u.GHz)

## 4. A report selects exact existing Results

`ReportSpec` neither solves nor searches for a latest result. It records exactly which typed Results the report presents.

In [ ]:
report = run.build_report(ReportSpec(inputs=(direct, evaluated)))
report.show()